# Metricas y conclusiones del laboratorio de integridad

Este notebook resume auditoria (`fsck`), consistencia origen-destino y tiempos del pipeline.


In [ ]:
from pathlib import Path
import pandas as pd

DT = '2026-02-23'
BASE = Path('..') if Path('..').exists() else Path('.')
OUT = BASE / 'outputs'
FSCK = OUT / 'fsck' / DT / 'fsck_summary.csv'
STEPS = OUT / 'metrics' / DT / 'step_times.csv'
REPL = OUT / 'metrics' / DT / 'replication_experiment.csv'
INV_TXT = OUT / 'inventory' / DT / 'inventory_summary.txt'

print('FSCK:', FSCK)
print('STEPS:', STEPS)
print('REPL:', REPL)
print('INV :', INV_TXT)


In [ ]:
df_fsck = pd.read_csv(FSCK)
df_steps = pd.read_csv(STEPS)
df_repl = pd.read_csv(REPL)

inventory = {}
for line in INV_TXT.read_text(encoding='utf-8').splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        inventory[k] = v
df_inventory = pd.DataFrame([inventory])

df_fsck


In [ ]:
print('Resumen de consistencia origen vs backup')
display(df_inventory)

print('Tiempos de pipeline (segundos)')
display(df_steps.sort_values('duration_seconds', ascending=False))

print('Experimento de replicacion (setrep -w)')
display(df_repl.sort_values('replication'))


## Conclusiones

1. La auditoria `fsck` muestra `HEALTHY` en `/data` y `/backup`, con `Corrupt/Missing/Under-replicated = 0`.
2. La validacion de inventario indica `ok_count=2` y sin diferencias de tamano entre origen y backup.
3. En este entorno, las operaciones mas costosas del pipeline fueron ingesta y simulacion/recuperacion de incidente.
4. La replicacion `3` maximiza tolerancia a fallos, pero incrementa I/O y red respecto a `2`.
5. Recomendacion operativa: usar replicacion `2` para cargas no criticas y `3` para datos sensibles o de alto impacto.
6. Frecuencia sugerida de auditoria: diaria para rutas criticas y semanal para historico estable.
